# Taller 1: Econometría

- **Profesor:** Francisco Alfaro Medina
- **Ayudantes:** Krischnna Cortez y Karen Rojas


### Instrucciones

- Dispone de **60 minutos** para completar los **100 puntos** del taller.
- Cuide la presentación y redacción de sus respuestas.
- Puede utilizar su computador y los apuntes de clase y ayudantía.
- Debe entregar un archivo **PDF** y un **R script** (extensión `.R`).

> ⚠️ **Importante para Colab:** Este notebook usa un kernel de R. Si abre este archivo en Google Colab, seleccione **Runtime → Change runtime type → R** antes de ejecutar cualquier celda.

---
# Sección 1: Datos Aleatorios *(30 puntos)*

En esta sección trabajaremos con un dataset **simulado** de 50 alumnos de la USM. El dataset contiene las siguientes variables:

| Variable | Descripción |
|---|---|
| `hrs_sueno` | Horas de sueño promedio en el último mes |
| `profesor_part` | Si recibió ayuda de profesor particular (0/1) |
| `media_sem_pasado` | Promedio de notas del semestre anterior |
| `tiempo_est` | Horas de estudio dedicadas |
| `asistencia` | Porcentaje de asistencia a clases |
| `nivel_socioec` | Nivel socioeconómico (1 al 5) |
| `notas` | **Variable dependiente** — nota del alumno |

## Pregunta 1.1 — Generar el dataset *(6 pts.)*

Antes de ejecutar el código, **cambie la semilla** según la primera letra de su apellido:

| A–E | F–J | K–O | P–T | U–Z |
|:---:|:---:|:---:|:---:|:---:|
| 123 | 456 | 789 | 101112 | 131415 |

Reemplace el valor en `set.seed(...)` antes de continuar.

In [1]:
# -------------------------------------------------------
# Pregunta 1.1: Generar el dataframe "datos"
# Cambie la semilla según la primera letra de su apellido
# A-E: 123 | F-J: 456 | K-O: 789 | P-T: 101112 | U-Z: 131415
# -------------------------------------------------------

set.seed(789)  # <-- CAMBIE ESTE VALOR SEGÚN SU APELLIDO

datos <- data.frame(
  hrs_sueno        = round(runif(50, min = 5,  max = 10),  1),
  profesor_part    = sample(c(0, 1), 50, replace = TRUE),
  media_sem_pasado = round(runif(50, min = 60, max = 100), 1),
  tiempo_est       = round(runif(50, min = 1,  max = 8),   1),
  asistencia       = round(runif(50, min = 60, max = 100), 1),
  nivel_socioec    = sample(1:5, 50, replace = TRUE)
)

# Calcular notas con ponderaciones definidas
datos$notas <- 30 +
  datos$hrs_sueno        * 1.5  +
  datos$profesor_part    * 3    +
  datos$media_sem_pasado * 0.2  +
  datos$tiempo_est       * 2    +
  datos$asistencia       * 0.15 +
  datos$nivel_socioec    * 2    +
  rnorm(50, mean = 0, sd = 5)

# Asegurar rango entre 20 y 100
datos$notas <- pmax(pmin(datos$notas, 100), 20)

# Vista rápida del dataset
cat("Dimensiones del dataset:", nrow(datos), "filas x", ncol(datos), "columnas\n")
head(datos)

Dimensiones del dataset: 50 filas x 7 columnas


,hrs_sueno,profesor_part,media_sem_pasado,tiempo_est,asistencia,nivel_socioec,notas
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>
1,8.5,0,79.2,3.8,71.7,2,78.60754
2,5.5,0,72.6,6.1,63.7,5,80.41091
3,5.1,1,62.6,7.5,73.8,4,93.97003
4,8.0,1,95.6,4.2,68.6,3,84.18621
5,7.5,1,91.5,6.5,88.8,2,90.67901
6,5.1,0,66.5,1.2,97.1,3,70.89307


## Pregunta 1.2 — Redondear notas *(2 pts.)*

Redondee la variable `notas` a **1 decimal**.

In [5]:
# Pregunta 1.2: Redondear notas a 1 decimal
datos$notas <- round(datos$notas,1)
head(datos)


,hrs_sueno,profesor_part,media_sem_pasado,tiempo_est,asistencia,nivel_socioec,notas
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>
1,8.5,0,79.2,3.8,71.7,2,78.6
2,5.5,0,72.6,6.1,63.7,5,80.4
3,5.1,1,62.6,7.5,73.8,4,94.0
4,8.0,1,95.6,4.2,68.6,3,84.2
5,7.5,1,91.5,6.5,88.8,2,90.7
6,5.1,0,66.5,1.2,97.1,3,70.9


## Pregunta 1.3 — Estimadores β via álgebra matricial *(8 pts.)*

Calcule los estimadores MCO **manualmente**, usando la fórmula matricial:

$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$$

**Sin usar** la función `lm()` de R.

In [10]:
#Estimadores
# Y (variable dependiente)
Y <- as.matrix(datos$notas)

# matriz X (con intercepto)
X <- as.matrix(cbind(
  1,
  datos$hrs_sueno,
  datos$profesor_part,
  datos$media_sem_pasado,
  datos$tiempo_est,
  datos$asistencia,
  datos$nivel_socioec
))

# Transpuesta de X
Xt <- t(X)

# Producto X'X
XtX <- Xt %*% X

# Inversa de (X'X)
XtX_inv <- solve(XtX)

# Producto X'Y
XtY <- Xt %*% Y

# Estimador beta
beta <- XtX_inv %*% XtY

# Mostrar resultados
beta

30.3078768
1.3003464
3.8976675
0.2586088
1.6779676
0.1146847
2.6659308


## Pregunta 1.4 — Modelo con `lm()` e interpretación *(8 pts.)*

Genere el modelo de regresión múltiple usando la función `lm()` y obtenga el resumen con `summary()`.  
Luego, **interprete cada coeficiente** en el espacio indicado.

> 💡 **Tip:** Los β de `lm()` deben coincidir con los calculados manualmente en la pregunta anterior.

In [11]:
# Pregunta 1.4: Modelo con lm() y summary
modelo <- lm(notas ~ hrs_sueno + profesor_part + media_sem_pasado +
               tiempo_est + asistencia + nivel_socioec,
             data = datos)

summary(modelo)


Call:
lm(formula = notas ~ hrs_sueno + profesor_part + media_sem_pasado + 
    tiempo_est + asistencia + nivel_socioec, data = datos)

Residuals:
    Min      1Q  Median      3Q     Max 
-9.2064 -3.6296  0.3241  3.4046  7.6236 

Coefficients:
                 Estimate Std. Error t value Pr(>|t|)    
(Intercept)      30.30788    8.83385   3.431 0.001340 ** 
hrs_sueno         1.30035    0.57855   2.248 0.029787 *  
profesor_part     3.89767    1.50901   2.583 0.013282 *  
media_sem_pasado  0.25861    0.06483   3.989 0.000253 ***
tiempo_est        1.67797    0.36369   4.614 3.55e-05 ***
asistencia        0.11468    0.06709   1.709 0.094603 .  
nivel_socioec     2.66593    0.59447   4.485 5.37e-05 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 4.98 on 43 degrees of freedom
Multiple R-squared:  0.6069,	Adjusted R-squared:  0.5521 
F-statistic: 11.06 on 6 and 43 DF,  p-value: 1.973e-07


Intercepto (B0=30,3) representa la nota esperada cuando las demas variables son cero.
Horas de sueño (B1 = 1,3):un aumento de una hora de sueño incrementa la nota en aproximadamente 1,3 puntos en promedio.
Profesor particular (B2 =3.9): los alumnos que tienen profesor particular obtienen en promedio 3,9  puntos más que aquellos que no tienen.
Promedio semestre pasado (B3 = 0,3): Un aumento de un punto en el promedio del semestre anterior incrementa la nota actual en aproximadamente 0,3 puntos.
Tiempo de estudio (B4 =1,7): Por cada hora adicional de estudio, la nota aumenta en promedio 1,7 puntos.
Asistencia (B5 = 0,1): Un aumento de un 1% en la asistencia a clases incrementa la nota en aproximadamente 0,1 puntos.
Nivel socioeconómico (B6 = 2,7): Un aumento en una unidad del nivel socioeconómico incrementa la nota en aproximadamente 2,7 puntos.



## Pregunta 1.5 — Relación entre betas y ponderaciones del código *(6 pts.)*

Analice la relación entre los coeficientes estimados (β̂) y las ponderaciones reales usadas en el código del Anexo para generar las notas.



In [12]:
# Pregunta 1.5: Comparación entre betas estimados y ponderaciones reales
# Calcular las notas con ajustes en la ponderación
datos$notas <- 30 + # Base más baja para empezar desde un mínimo más realista
datos$hrs_sueno * 1.5 + # Reducir el impacto del sueño
datos$profesor_part * 3 + # Menos puntos adicionales por participación del profesor
datos$media_sem_pasado * 0.2 + # Menor peso de la media semestral pasada
datos$tiempo_est * 2 + # Ajuste en tiempo de estudio
datos$asistencia * 0.15 + # Menor impacto de la asistencia
datos$nivel_socioec * 2 + # Menos impacto del nivel socioeconómico
rnorm(50, mean = 0, sd = 5) # Menor desviación estándar para reducir variabilidad
# Asegurar que las notas estén entre 20 y 100
datos$notas <- pmax(pmin(datos$notas, 100), 20)



# Sección 2: Wooldridge *(70 puntos)*

Para esta sección utilizaremos el paquete `wooldridge`, que contiene bases de datos clásicas de econometría.

In [13]:
# Instalar y cargar el paquete wooldridge (solo necesario la primera vez en Colab)
if (!require(wooldridge)) install.packages("wooldridge")
library(wooldridge)

Loading required package: wooldridge

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘wooldridge’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



---
## 2A. Base `wage1` *(30 puntos)*

El modelo de regresión poblacional a estimar es:

$$wage = \beta_0 + \beta_1\, educ + \beta_2\, exper + \beta_3\, tenure + u$$

Donde:
- `wage` = salario por hora (USD)
- `educ` = años de escolaridad
- `exper` = años de experiencia laboral
- `tenure` = años en el trabajo actual

In [14]:
# Cargar y limpiar la base wage1
data("wage1")
wage1 <- na.omit(wage1)

### Pregunta 2A.1 — Estimadores MCO e interpretación *(8 pts.)*

Estime el modelo completo e interprete los resultados.

In [15]:
# Pregunta 2A.1: Modelo de regresión múltiple con wage1
modelo_wage <- lm(wage ~ educ + exper + tenure, data = wage1)
summary(modelo_wage)


Call:
lm(formula = wage ~ educ + exper + tenure, data = wage1)

Residuals:
    Min      1Q  Median      3Q     Max 
-7.6068 -1.7747 -0.6279  1.1969 14.6536 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept) -2.87273    0.72896  -3.941 9.22e-05 ***
educ         0.59897    0.05128  11.679  < 2e-16 ***
exper        0.02234    0.01206   1.853   0.0645 .  
tenure       0.16927    0.02164   7.820 2.93e-14 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 3.084 on 522 degrees of freedom
Multiple R-squared:  0.3064,	Adjusted R-squared:  0.3024 
F-statistic: 76.87 on 3 and 522 DF,  p-value: < 2.2e-16


Los resultados indican que los 3 coeficientes ( años de escolaridad, experienca y trabajo actual) son positivos, lo que sugiere que en promedio todos influyen en un aumento o incremento del salario.  

### Pregunta 2A.2 — ¿Los signos son los esperados? *(10 pts.)*

Antes de ver los resultados, reflexione: ¿qué signo debería tener cada coeficiente económicamente?

In [ ]:
# Pregunta 2A.2: Revisar signos de los coeficientes


Sí, los coeficientes presentan los signos esperados. Se espera que la educación, la experiencia laboral y la antigüedad en el trabajo tengan efectos positivos sobre el salario, ya que aumentan la productividad del trabajador. Por lo tanto, los resultados del modelo son coherentes con la teoría económica.

### Pregunta 2A.3 — Comparar R² y R² ajustado *(12 pts.)*

Estime un segundo modelo usando solo `educ` y `tenure`, y compare el ajuste con el modelo completo.

In [16]:
# Pregunta 2A.3: Modelo reducido (sin exper)
modelo_wage2 <- lm(wage ~ educ + tenure, data = wage1)
summary(modelo_wage2)


Call:
lm(formula = wage ~ educ + tenure, data = wage1)

Residuals:
    Min      1Q  Median      3Q     Max 
-8.1438 -1.7288 -0.6372  1.2575 14.7482 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept) -2.22162    0.64015   -3.47 0.000563 ***
educ         0.56914    0.04881   11.66  < 2e-16 ***
tenure       0.18958    0.01871   10.13  < 2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 3.092 on 523 degrees of freedom
Multiple R-squared:  0.3019,	Adjusted R-squared:  0.2992 
F-statistic: 113.1 on 2 and 523 DF,  p-value: < 2.2e-16


---
## 2B. Base `attend` *(40 puntos)*

En esta sección analizamos los determinantes del rendimiento en el examen final de un curso universitario.

### Pregunta 2B.1 — Cargar la base `attend` *(4 pts.)*

In [19]:
# Pregunta 2B.1: Cargar base attend
data("attend")
attend <- na.omit(attend)

### Pregunta 2B.2 — Selección de variables *(4 pts.)*

Subseleccione las siguientes variables:

| Variable | Descripción |
|---|---|
| `attend` | Clases asistidas de un total de 32 |
| `termGPA` | Promedio de notas durante el período |
| `priGPA` | Promedio acumulado antes del período |
| `ACT` | Puntaje en el examen ACT |
| `final` | **Variable dependiente** — puntaje del examen final |
| `hwrte` | Porcentaje de tareas entregadas |
| `frosh` | =1 si es estudiante de primer año |
| `soph` | =1 si es estudiante de segundo año |

In [20]:
# Pregunta 2B.2: Subselección de variables
datos2 <- attend[, c("attend", "termGPA", "priGPA", "ACT", "final", "hwrte", "frosh", "soph")]


### Pregunta 2B.3 — Modelo de regresión múltiple completo *(10 pts.)*

Estime un modelo donde la variable dependiente es `final` y las independientes son todas las demás variables del subconjunto.

In [21]:
# Pregunta 2B.3: Modelo completo con attend_new
summary(datos2)

     attend         termGPA          priGPA           ACT       
 Min.   : 2.00   Min.   :0.000   Min.   :0.857   Min.   :13.00  
 1st Qu.:24.00   1st Qu.:2.150   1st Qu.:2.200   1st Qu.:20.00  
 Median :28.00   Median :2.680   Median :2.560   Median :22.00  
 Mean   :26.28   Mean   :2.614   Mean   :2.592   Mean   :22.48  
 3rd Qu.:30.00   3rd Qu.:3.120   3rd Qu.:2.950   3rd Qu.:25.00  
 Max.   :32.00   Max.   :4.000   Max.   :3.930   Max.   :32.00  
     final           hwrte            frosh           soph       
 Min.   :10.00   Min.   : 12.50   Min.   :0.00   Min.   :0.0000  
 1st Qu.:22.00   1st Qu.: 87.50   1st Qu.:0.00   1st Qu.:0.0000  
 Median :26.00   Median :100.00   Median :0.00   Median :1.0000  
 Mean   :25.89   Mean   : 87.91   Mean   :0.23   Mean   :0.5801  
 3rd Qu.:29.00   3rd Qu.:100.00   3rd Qu.:0.00   3rd Qu.:1.0000  
 Max.   :39.00   Max.   :100.00   Max.   :1.00   Max.   :1.0000  

### Pregunta 2B.4 — Bondad de ajuste *(6 pts.)*

Interprete el **R²** y el **R² ajustado** del modelo anterior.

In [ ]:
# Pregunta 2B.4: Extraer métricas de bondad de ajuste

### Pregunta 2B.5 — Modelo reducido (excluir variables no significativas) *(10 pts.)*

Genere un nuevo modelo excluyendo las variables con **p-value > 0.05** en el modelo anterior.

In [ ]:
# Identificar variables significativas (p-value <= 0.05)

In [ ]:
# Pregunta 2B.5: Modelo reducido (solo variables significativas)
# Variables significativas identificadas: termGPA, ACT, soph
# (ajuste según los resultados de su modelo)

### Pregunta 2B.6 — Comparación de modelos *(6 pts.)*

Compare el modelo completo y el modelo reducido en términos de **R² ajustado**.

In [ ]:
# Pregunta 2B.6: Comparación final de modelos